# Partie A : LangChain

Agent simple, mémoire, tools, tools prédéfinis, middlewares et chatbot RAG agentique, avec **Azure OpenAI**.

## A.0 Préparation

- Charger le fichier `.env` avec `load_dotenv(override=True)`
- Créer **une seule fois** le modèle `llm = AzureChatOpenAI(...)` (avec `timeout=30`), réutilisé dans toute la partie A
- Tester la connexion avec `llm.invoke("Bonjour").content`

In [10]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

True

In [11]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("Azure_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0,
    timeout=30,
)

In [12]:
llm.invoke("Bonjour, présente-toi en une phrase.").content

'Bonjour ! Je suis ChatGPT, une intelligence artificielle conçue pour t’aider à répondre à tes questions et à discuter de nombreux sujets.'

## A.1 Agent simple

- Importer `create_agent` depuis `langchain.agents`
- Créer un agent **sans tools** : `create_agent(model=llm, system_prompt="...")`
- L'appeler avec `agent.invoke({"messages": [HumanMessage("...")]})`
- Afficher la réponse : `response["messages"][-1].content`
- **Configuration du modèle** : faire varier `temperature` et `max_tokens` et observer l'effet

## A.2 Mémoire de l'agent

- Importer `InMemorySaver` depuis `langgraph.checkpoint.memory`
- Créer l'agent avec `checkpointer=InMemorySaver()`
- Appeler l'agent avec `config={"configurable": {"thread_id": "1"}}`
- **Test** :
  - « Je m'appelle X » puis « Comment je m'appelle ? » avec le **même** `thread_id` → l'agent se souvient
  - Même question avec un **autre** `thread_id` → il ne sait pas

## A.3 Ajouter des tools à l'agent

- Écrire des fonctions avec le décorateur `@tool` (`from langchain.tools import tool`)
- Mettre des **annotations de type** et une **docstring claire** : c'est elle qui permet au LLM de choisir le tool
- Exemples : `get_weather(city)`, `get_employee_info(name)` (données factices)
- Créer l'agent : `create_agent(model=llm, tools=[...])`
- Afficher **tous** les `messages` de la réponse pour montrer les `tool_calls` et les `ToolMessage`

## A.4 Tools prédéfinis

| Tool | Import | Remarque |
|---|---|---|
| **DuckDuckGo** | `from ddgs import DDGS` | Placé dans un `@tool`, aucune clé nécessaire |
| **Tavily** | `from langchain_tavily import TavilySearch` | `TavilySearch(max_results=3)`, utilise `TAVILY_API_KEY` |
| **PythonREPL** | `from langchain_experimental.tools import PythonREPLTool` | L'agent exécute du Python (calculs, graphiques) |

- Montrer un agent qui **combine plusieurs tools** et choisit le bon selon la question
- **Attention** : `PythonREPLTool` exécute réellement du code sur la machine

## A.5 Middlewares

Import depuis `langchain.agents.middleware`, puis `create_agent(..., middleware=[...])`

| Middleware | Élément LangChain | Démonstration |
|---|---|---|
| **dynamic_model** | `@wrap_model_call` | Changer de modèle / de réglages selon la longueur de la conversation |
| **dynamic_prompt** | `@dynamic_prompt` | Adapter le system prompt au contexte (ex. utilisateur expert ou débutant) |
| **tool_error_handling** | `@wrap_tool_call` | Rattraper l'exception d'un tool et renvoyer un `ToolMessage` d'erreur lisible |
| **Guardrails** | `PIIMiddleware`, `@before_agent`, `@after_agent` | Masquer e-mail / carte bancaire, bloquer un mot interdit, vérifier la réponse |
| **Human in the Loop** | `HumanInTheLoopMiddleware` + `checkpointer` | Pause avant un tool sensible, `result["__interrupt__"]`, reprise avec `Command(resume=...)` |

- **Bonus** : Structured Output avec un `BaseModel` Pydantic

## A.6 Chatbot RAG agentique

1. **Charger** un document PDF avec `PyPDFLoader` (`langchain_community.document_loaders`)
2. **Découper** avec `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)`
3. **Embeddings** avec `AzureOpenAIEmbeddings(...)`
4. **Indexer** dans `FAISS.from_documents(chunks, embeddings)`
5. **Créer le tool** : `create_retriever_tool(vector_store.as_retriever(), "nom", "description")`
6. **Créer l'agent** avec ce tool, une mémoire (`InMemorySaver`) et un system prompt qui demande de **citer les sources**
7. **Tester** avec 3 ou 4 questions qui se suivent : l'agent cherche dans le document **et** se souvient de la conversation